# 🔐 CYBER CRIME INVESTIGATION
## Using Data Analysis to Catch an Insider Threat

---

> **CASE FILE #2026-XRAY**  
> *Last night, confidential company data was leaked.*  
> *We have system logs from 50 employees.*  
> **Your mission: Use Data Analysis to identify the suspect.**

---

## 📦 PHASE 1 — Setup: Import Our Investigation Tools

Every good detective needs tools. In data science, our tools are Python libraries.

| Library | What it does for us |
|---------|--------------------|
| `pandas` | Read and filter suspect data |
| `matplotlib` | Draw charts to spot patterns |
| `seaborn` | Beautiful heatmaps and graphs |
| `numpy` | Math for suspicion scoring |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Set the visual style — dark theme for our investigation dashboard
plt.style.use('dark_background')
sns.set_palette('Reds_r')

print('✅ Investigation tools loaded successfully.')
print('🔍 Ready to analyze suspects.')

---

## 📋 PHASE 2 — Load the Evidence: Read the Employee Logs

We have system logs for **6 employees** who had access to the server last night.

Let's load the data and take our first look at the suspects.

> 💬 *Question 1: "Without any analysis — who looks suspicious to you just by reading this?"*

In [ ]:
# Load the employee activity logs
df = pd.read_csv('employee_logs.csv')

print('📁 CASE FILE LOADED')
print(f'   Employees under investigation : {len(df)}')
print(f'   Data points per employee      : {len(df.columns)}')
print(f'   Columns (evidence types)      : {list(df.columns)}')
print()

# Display the full evidence table
print('🗂️  RAW EVIDENCE TABLE:')
df

---

## 🔬 PHASE 3 — Data Analysis: Understand the Numbers

Before we filter suspects, we need to understand what "normal" looks like.  
Anything far above average is an **anomaly** — and anomalies are clues.

> 💬 *Question: "What is the average number of files opened? What's the maximum? Why is the max suspicious?"*

In [ ]:
# Statistical summary — our baseline for 'normal' behavior
print('📊 STATISTICAL SUMMARY — Baseline Normal Behavior')
print('=' * 55)

numeric_cols = [
    'Login_Hour', 'Files_Opened', 'Files_Deleted',
    'Failed_Logins', 'Typing_Speed_WPM', 'Data_Transfer_MB',
    'Emails_Sent', 'External_Emails'
]

stats = df[numeric_cols].describe().round(2)
stats

In [ ]:
# Highlight anomalies — values more than 2x the average
print('🚨 ANOMALY DETECTION — Values far above average:')
print('=' * 55)

for col in ['Data_Transfer_MB', 'Files_Opened', 'Failed_Logins', 'External_Emails', 'Files_Deleted']:
    avg = df[col].mean()
    for _, row in df.iterrows():
        if row[col] > avg * 2:
            print(f'   ⚠️  {row["Employee"]:6} | {col:20} = {row[col]} (avg is {avg:.1f})')

---

## 🕵️ PHASE 4 — Filtering: Narrow Down Suspects

A detective doesn't look at everyone equally.  
We **filter** to focus only on employees showing suspicious behavior.

> 💬 *"We are not writing code. We are eliminating suspects."*

In [ ]:
# Filter 1 — Who was active at night?
print('🌙 FILTER 1: Late Night Activity')
night_users = df[df['Late_Night_Activity'] == 'Yes']
print(f'   {len(night_users)} suspect(s) were active at night:')
print(night_users[['Employee', 'Department', 'Login_Hour', 'Logout_Hour']].to_string(index=False))
print()

In [ ]:
# Filter 2 — Who used a USB device?
print('💾 FILTER 2: USB Device Inserted')
usb_users = df[df['USB_Inserted'] == 'Yes']
print(f'   {len(usb_users)} suspect(s) used a USB drive:')
print(usb_users[['Employee', 'Department', 'USB_Inserted', 'Data_Transfer_MB']].to_string(index=False))
print()

In [ ]:
# Filter 3 — Who had too many failed logins?
print('🔐 FILTER 3: Multiple Failed Login Attempts (> 3)')
failed_users = df[df['Failed_Logins'] > 3]
print(f'   {len(failed_users)} suspect(s) had excessive failed logins:')
print(failed_users[['Employee', 'Department', 'Failed_Logins']].to_string(index=False))
print()

In [ ]:
# Filter 4 — Combined: Night + USB + High Transfer
print('🚨 FILTER 4: COMBINED — Night + USB + Transfer > 500MB')
primary_suspects = df[
    (df['Late_Night_Activity'] == 'Yes') &
    (df['USB_Inserted'] == 'Yes') &
    (df['Data_Transfer_MB'] > 500)
]
print(f'   {len(primary_suspects)} PRIMARY SUSPECT(S) identified:')
print(primary_suspects[['Employee', 'Department', 'Login_Hour', 'Data_Transfer_MB', 'Failed_Logins']].to_string(index=False))

---

## 📊 PHASE 5 — Visualization: See the Evidence

Numbers are evidence. But **charts are proof you can see.**

We'll now plot 4 key evidence charts:
1. Files Opened per employee
2. Data Transfer spikes
3. Failed Login attempts
4. Suspicion Heatmap

> 💬 *"If a chart has one bar WAY taller than others — that's an anomaly. Anomalies are clues."*

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Sort employees by files opened
df_sorted = df.sort_values(by='Files_Opened', ascending=True)

# Dynamic figure height based on employee count
fig_height = max(8, len(df_sorted) * 0.35)

fig, ax = plt.subplots(figsize=(14, fig_height))

# Color logic
colors = [
    '#ff3366' if x > 80 else
    '#ffaa00' if x > 50 else
    '#00bfff'
    for x in df_sorted['Files_Opened']
]

# Horizontal bars
bars = ax.barh(
    df_sorted['Employee'],
    df_sorted['Files_Opened'],
    color=colors,
    edgecolor='white',
    linewidth=0.5
)

# Value labels
for bar, val in zip(bars, df_sorted['Files_Opened']):
    ax.text(
        val + 2,
        bar.get_y() + bar.get_height()/2,
        str(val),
        va='center',
        color='white',
        fontsize=9,
        fontweight='bold'
    )

# Average line
avg = df_sorted['Files_Opened'].mean()

ax.axvline(
    avg,
    color='#00ff88',
    linestyle='--',
    linewidth=2
)

# Title
ax.set_title(
    '🚨 Employee File Access Analysis',
    fontsize=22,
    color='white',
    pad=20,
    fontweight='bold'
)

# Axis labels
ax.set_xlabel(
    'Files Opened',
    fontsize=14,
    color='white',
    labelpad=10
)

ax.set_ylabel(
    'Employees',
    fontsize=14,
    color='white',
    labelpad=10
)

# Tick styling
ax.tick_params(axis='x', colors='white', labelsize=11)
ax.tick_params(axis='y', colors='white', labelsize=9)

# Background styling
ax.set_facecolor('#0a0f1a')
fig.patch.set_facecolor('#0a0f1a')

# Grid
ax.grid(
    axis='x',
    linestyle='--',
    alpha=0.2,
    color='white'
)

# Legend
normal_patch = mpatches.Patch(color='#00bfff', label='Normal')
warning_patch = mpatches.Patch(color='#ffaa00', label='Warning')
danger_patch = mpatches.Patch(color='#ff3366', label='Critical Suspicion')

legend = ax.legend(
    handles=[normal_patch, warning_patch, danger_patch],
    fontsize=11,
    facecolor='#111827',
    edgecolor='white',
    loc='lower right'
)

# Legend text color
for text in legend.get_texts():
    text.set_color('white')

# Remove unnecessary borders
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.spines['left'].set_color('white')
ax.spines['bottom'].set_color('white')

# Tight layout
plt.tight_layout()

# Save image
plt.savefig(
    'employee_file_analysis.png',
    dpi=200,
    bbox_inches='tight',
    facecolor=fig.get_facecolor()
)

plt.show()

print("💾 Saved: employee_file_analysis.png")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Sort by data transfer
df_sorted = df.sort_values(by='Data_Transfer_MB', ascending=True)

# Dynamic chart height
fig_height = max(8, len(df_sorted) * 0.35)

fig, ax = plt.subplots(figsize=(15, fig_height))

# Threat color logic
colors = [
    '#ff0033' if x > 900 else
    '#ffaa00' if x > 400 else
    '#00bfff'
    for x in df_sorted['Data_Transfer_MB']
]

# Horizontal bars
bars = ax.barh(
    df_sorted['Employee'],
    df_sorted['Data_Transfer_MB'],
    color=colors,
    edgecolor='white',
    linewidth=0.5
)

# Add transfer labels
for bar, val in zip(bars, df_sorted['Data_Transfer_MB']):
    ax.text(
        val + 15,
        bar.get_y() + bar.get_height()/2,
        f'{val} MB',
        va='center',
        color='white',
        fontsize=8,
        fontweight='bold'
    )

# Average transfer line
avg = df_sorted['Data_Transfer_MB'].mean()

ax.axvline(
    avg,
    color='#00ff88',
    linestyle='--',
    linewidth=2
)

# Title
ax.set_title(
    '📡 Network Data Transfer Monitoring',
    fontsize=22,
    color='white',
    pad=20,
    fontweight='bold'
)

# Axis labels
ax.set_xlabel(
    'Data Transfer (MB)',
    fontsize=14,
    color='white',
    labelpad=10
)

ax.set_ylabel(
    'Employees',
    fontsize=14,
    color='white',
    labelpad=10
)

# Tick styling
ax.tick_params(axis='x', colors='white', labelsize=11)
ax.tick_params(axis='y', colors='white', labelsize=8)

# Grid
ax.grid(
    axis='x',
    linestyle='--',
    alpha=0.2,
    color='white'
)

# Dark cyber background
ax.set_facecolor('#0a0f1a')
fig.patch.set_facecolor('#0a0f1a')

# Legend
normal_patch = mpatches.Patch(color='#00bfff', label='Normal Traffic')
warning_patch = mpatches.Patch(color='#ffaa00', label='Moderate Spike')
critical_patch = mpatches.Patch(color='#ff0033', label='Critical Threat')

legend = ax.legend(
    handles=[normal_patch, warning_patch, critical_patch],
    fontsize=11,
    facecolor='#111827',
    edgecolor='white',
    loc='lower right'
)

# White legend text
for text in legend.get_texts():
    text.set_color('white')

# Border styling
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.spines['left'].set_color('white')
ax.spines['bottom'].set_color('white')

# Layout optimization
plt.tight_layout()

# Save chart
plt.savefig(
    'chart2_data_transfer.png',
    dpi=220,
    bbox_inches='tight',
    facecolor=fig.get_facecolor()
)

plt.show()

print("💾 Saved: chart2_data_transfer.png")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Sort by failed logins
df_sorted = df.sort_values(by='Failed_Logins', ascending=True)

# Dynamic figure height
fig_height = max(8, len(df_sorted) * 0.35)

fig, ax = plt.subplots(figsize=(14, fig_height))

# Threat color mapping
colors = [
    '#ff0033' if x >= 8 else
    '#ffaa00' if x >= 4 else
    '#00bfff'
    for x in df_sorted['Failed_Logins']
]

# Horizontal bars
bars = ax.barh(
    df_sorted['Employee'],
    df_sorted['Failed_Logins'],
    color=colors,
    edgecolor='white',
    linewidth=0.5
)

# Add labels
for bar, val in zip(bars, df_sorted['Failed_Logins']):
    ax.text(
        val + 0.2,
        bar.get_y() + bar.get_height()/2,
        str(val),
        va='center',
        color='white',
        fontsize=9,
        fontweight='bold'
    )

# Security threshold line
threshold = 3

ax.axvline(
    threshold,
    color='#00ff88',
    linestyle='--',
    linewidth=2
)

# Title
ax.set_title(
    '🔐 Failed Login Attempt Analysis',
    fontsize=22,
    color='white',
    pad=20,
    fontweight='bold'
)

# Axis labels
ax.set_xlabel(
    'Failed Login Attempts',
    fontsize=14,
    color='white',
    labelpad=10
)

ax.set_ylabel(
    'Employees',
    fontsize=14,
    color='white',
    labelpad=10
)

# Tick styling
ax.tick_params(axis='x', colors='white', labelsize=11)
ax.tick_params(axis='y', colors='white', labelsize=8)

# Grid styling
ax.grid(
    axis='x',
    linestyle='--',
    alpha=0.2,
    color='white'
)

# Background styling
ax.set_facecolor('#0a0f1a')
fig.patch.set_facecolor('#0a0f1a')

# Legend
normal_patch = mpatches.Patch(color='#00bfff', label='Normal Activity')
warning_patch = mpatches.Patch(color='#ffaa00', label='Suspicious Activity')
critical_patch = mpatches.Patch(color='#ff0033', label='Possible Attack')

legend = ax.legend(
    handles=[normal_patch, warning_patch, critical_patch],
    fontsize=11,
    facecolor='#111827',
    edgecolor='white',
    loc='lower right'
)

# White legend text
for text in legend.get_texts():
    text.set_color('white')

# Border styling
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.spines['left'].set_color('white')
ax.spines['bottom'].set_color('white')

# Tight layout
plt.tight_layout()

# Save image
plt.savefig(
    'chart3_failed_logins.png',
    dpi=220,
    bbox_inches='tight',
    facecolor=fig.get_facecolor()
)

plt.show()

print("💾 Saved: chart3_failed_logins.png")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Metrics used in analysis
heatmap_cols = [
    'Files_Opened',
    'Files_Deleted',
    'Failed_Logins',
    'Data_Transfer_MB',
    'External_Emails',
    'Typing_Speed_WPM'
]

# Prepare data
heatmap_data = df.set_index('Employee')[heatmap_cols]

# Create a suspicion score for sorting
heatmap_data['Threat_Score'] = (
    heatmap_data['Failed_Logins'] * 3 +
    heatmap_data['Files_Deleted'] * 2 +
    heatmap_data['External_Emails'] * 1.5 +
    heatmap_data['Data_Transfer_MB'] * 0.02
)

# Sort by threat score
heatmap_data = heatmap_data.sort_values(
    by='Threat_Score',
    ascending=False
)

# Remove helper column from heatmap
threat_scores = heatmap_data['Threat_Score']
heatmap_data = heatmap_data.drop(columns=['Threat_Score'])

# Normalize values (0–1)
heatmap_norm = (
    heatmap_data - heatmap_data.min()
) / (
    heatmap_data.max() - heatmap_data.min()
)

# Dynamic figure height
fig_height = max(10, len(heatmap_data) * 0.35)

# Create figure
fig, ax = plt.subplots(figsize=(16, fig_height))

# Heatmap
sns.heatmap(
    heatmap_norm,
    cmap='inferno',
    linewidths=0.3,
    linecolor='#111827',
    annot=heatmap_data,
    fmt='.0f',
    annot_kws={
        'size': 7,
        'color': 'white'
    },
    cbar_kws={
        'label': 'Threat Intensity'
    },
    ax=ax
)

# Titles
ax.set_title(
    '🔥 AI Threat Intelligence Heatmap',
    fontsize=24,
    color='white',
    pad=25,
    fontweight='bold'
)

# Axis styling
ax.set_xlabel(
    '',
    color='white'
)

ax.set_ylabel(
    'Employees',
    fontsize=14,
    color='white',
    labelpad=10
)

# Tick styling
ax.tick_params(
    axis='x',
    colors='white',
    labelsize=11,
    rotation=15
)

ax.tick_params(
    axis='y',
    colors='white',
    labelsize=8
)

# Background
ax.set_facecolor('#0a0f1a')
fig.patch.set_facecolor('#0a0f1a')

# Colorbar styling
cbar = ax.collections[0].colorbar
cbar.ax.yaxis.label.set_color('white')
cbar.ax.tick_params(colors='white')

# Layout optimization
plt.tight_layout()

# Save chart
plt.savefig(
    'chart4_threat_heatmap.png',
    dpi=220,
    bbox_inches='tight',
    facecolor=fig.get_facecolor()
)

# Show chart
plt.show()

print("💾 Saved: chart4_threat_heatmap.png")
print()
print("🚨 Employees near the TOP with the brightest patterns are the highest-risk suspects.")

---

## 🤖 PHASE 6 — AI Suspicion Scoring System

> *"This is where Data Analysis becomes Artificial Intelligence."*

We'll now build a **weighted scoring model** — the same concept used in:
- Credit scoring (banks)
- Fraud detection (Visa, Mastercard)
- Recommendation engines (Netflix, Spotify)

Each suspicious behavior gets a **weight** based on how serious it is.  
The employee with the **highest total score = most likely culprit.**

| Factor | Weight | Reason |
|--------|--------|--------|
| Failed Logins | × 10 | Strong hacking signal |
| Data Transfer MB | × 0.05 | Volume-based exfiltration |
| Files Opened | × 0.5 | Unauthorized access |
| Files Deleted | × 8 | Covering tracks |
| External Emails | × 3 | Sending data outside |
| Night Activity | +20 | Off-hours = suspicious |
| USB Inserted | +15 | Physical exfiltration |

In [ ]:
# ── SUSPICION SCORE FORMULA ──────────────────────────────
# You can change these weights and re-run to see scores update!

W_FAILED_LOGINS   = 10    # weight: failed login attempts
W_DATA_TRANSFER   = 0.05  # weight: megabytes transferred
W_FILES_OPENED    = 0.5   # weight: files accessed
W_FILES_DELETED   = 8     # weight: files deleted (covering tracks)
W_EXTERNAL_EMAILS = 3     # weight: emails sent outside company
BONUS_NIGHT       = 20    # flat bonus: late night activity
BONUS_USB         = 15    # flat bonus: USB device used

df['Suspicion_Score'] = (
    df['Failed_Logins']   * W_FAILED_LOGINS   +
    df['Data_Transfer_MB']* W_DATA_TRANSFER   +
    df['Files_Opened']    * W_FILES_OPENED    +
    df['Files_Deleted']   * W_FILES_DELETED   +
    df['External_Emails'] * W_EXTERNAL_EMAILS +
    (df['Late_Night_Activity'] == 'Yes').astype(int) * BONUS_NIGHT +
    (df['USB_Inserted'] == 'Yes').astype(int) * BONUS_USB
).round(1)

# Rank from most to least suspicious
ranked = df[['Employee', 'Department', 'Suspicion_Score']].sort_values(
    'Suspicion_Score', ascending=False
).reset_index(drop=True)

ranked.index = ranked.index + 1  # rank starts at 1
ranked.index.name = 'Rank'

print('🎯 SUSPICION SCORE RANKINGS')
print('=' * 45)
ranked

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Sort highest suspicion first
ranked = ranked.sort_values(
    by='Suspicion_Score',
    ascending=True
)

# Dynamic chart height
fig_height = max(10, len(ranked) * 0.35)

fig, ax = plt.subplots(figsize=(16, fig_height))

# Threat color logic
score_colors = []

for score in ranked['Suspicion_Score']:
    if score >= 90:
        score_colors.append('#ff0033')   # Critical
    elif score >= 70:
        score_colors.append('#ff6633')   # High
    elif score >= 40:
        score_colors.append('#ffaa00')   # Medium
    else:
        score_colors.append('#00bfff')   # Low

# Horizontal bars
bars = ax.barh(
    ranked['Employee'],
    ranked['Suspicion_Score'],
    color=score_colors,
    edgecolor='white',
    linewidth=0.6
)

# Add score labels
for bar, val in zip(bars, ranked['Suspicion_Score']):
    ax.text(
        val + 1,
        bar.get_y() + bar.get_height()/2,
        f'{val:.0f}',
        va='center',
        color='white',
        fontsize=9,
        fontweight='bold'
    )

# Highlight TOP suspect
top_employee = ranked.iloc[-1]['Employee']
top_score = ranked.iloc[-1]['Suspicion_Score']

ax.text(
    top_score * 0.5,
    len(ranked) - 1,
    '🚨 PRIMARY SUSPECT',
    color='white',
    fontsize=14,
    fontweight='bold',
    ha='center',
    va='center',
    bbox=dict(
        facecolor='#ff0033',
        edgecolor='white',
        boxstyle='round,pad=0.5'
    )
)

# Title
ax.set_title(
    '🎯 AI Threat Ranking System',
    fontsize=24,
    color='white',
    pad=25,
    fontweight='bold'
)

# Axis labels
ax.set_xlabel(
    'Suspicion Score',
    fontsize=14,
    color='white',
    labelpad=10
)

ax.set_ylabel(
    'Employees',
    fontsize=14,
    color='white',
    labelpad=10
)

# Tick styling
ax.tick_params(axis='x', colors='white', labelsize=11)
ax.tick_params(axis='y', colors='white', labelsize=8)

# Grid
ax.grid(
    axis='x',
    linestyle='--',
    alpha=0.2,
    color='white'
)

# Background
ax.set_facecolor('#0a0f1a')
fig.patch.set_facecolor('#0a0f1a')

# Legend
low_patch = mpatches.Patch(color='#00bfff', label='Low Risk')
medium_patch = mpatches.Patch(color='#ffaa00', label='Medium Risk')
high_patch = mpatches.Patch(color='#ff6633', label='High Risk')
critical_patch = mpatches.Patch(color='#ff0033', label='Critical Threat')

legend = ax.legend(
    handles=[
        low_patch,
        medium_patch,
        high_patch,
        critical_patch
    ],
    fontsize=11,
    facecolor='#111827',
    edgecolor='white',
    loc='lower right'
)

# White legend text
for text in legend.get_texts():
    text.set_color('white')

# Border styling
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.spines['left'].set_color('white')
ax.spines['bottom'].set_color('white')

# Layout
plt.tight_layout()

# Save chart
plt.savefig(
    'chart5_ai_threat_ranking.png',
    dpi=240,
    bbox_inches='tight',
    facecolor=fig.get_facecolor()
)

# Show chart
plt.show()

print("💾 Saved: chart5_ai_threat_ranking.png")

print()
print("🚨 AI INVESTIGATION COMPLETE")
print(f"🎯 Primary Suspect: {top_employee}")
print(f"⚠️ Threat Score: {top_score:.2f}")

---

## 🚨 PHASE 7 — FINAL REVEAL

Based on all our evidence:
- Statistical analysis
- Filtering
- Visualization
- AI scoring model

**The data has spoken. The suspect is...**

In [ ]:
# ── FINAL REVEAL ─────────────────────────────────────────
top_suspect = ranked.iloc[0]['Employee']
top_score   = ranked.iloc[0]['Suspicion_Score']
suspect_row = df[df['Employee'] == top_suspect].iloc[0]

print('=' * 60)
print('  ⚠️   C R I M I N A L   I D E N T I F I E D   ⚠️')
print('=' * 60)
print(f'  NAME       : {top_suspect}')
print(f'  DEPARTMENT : {suspect_row["Department"]}')
print(f'  SCORE      : {top_score} / 100 (CRITICAL THREAT)')
print('=' * 60)
print()
print('📋 EVIDENCE AGAINST', top_suspect.upper())
print('-' * 40)
print(f'  🌙 Logged in at {suspect_row["Login_Hour"]}:00 AM  (late night access)')
print(f'  💾 USB drive inserted              (physical exfiltration)')
print(f'  📡 {suspect_row["Data_Transfer_MB"]} MB of data transferred   (massive leak)')
print(f'  📂 {suspect_row["Files_Opened"]} files opened             (unauthorized access)')
print(f'  🗑️  {suspect_row["Files_Deleted"]} files deleted            (covering tracks)')
print(f'  🔐 {suspect_row["Failed_Logins"]} failed login attempts    (hacking attempt)')
print(f'  📧 {suspect_row["External_Emails"]} emails sent externally   (data exfiltration)')
print(f'  💰 Accessed Finance records        (sensitive data breach)')
print()
print('  🤖 AI VERDICT: 95% probability of insider threat.')
print('  ✅ Case closed. Data Analysis identified the criminal.')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Metrics for comparison
compare_cols = [
    'Data_Transfer_MB',
    'Files_Opened',
    'Failed_Logins'
]

titles = [
    '📡 Data Transfer',
    '📂 Files Opened',
    '🔐 Failed Logins'
]

# Get suspect data
suspect_data = df[df['Employee'] == top_suspect].iloc[0]

# Company averages
avg_data = df[compare_cols].mean()

# Figure setup
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Background styling
fig.patch.set_facecolor('#0a0f1a')

# Main title
fig.suptitle(
    f'🚨 Final AI Evidence Report — {top_suspect}',
    fontsize=22,
    color='white',
    fontweight='bold',
    y=1.02
)

# Chart loop
for ax, col, title in zip(axes, compare_cols, titles):

    values = [
        avg_data[col],
        suspect_data[col]
    ]

    labels = [
        'Company Avg',
        top_suspect
    ]

    colors = [
        '#00bfff',
        '#ff0033'
    ]

    bars = ax.bar(
        labels,
        values,
        color=colors,
        edgecolor='white',
        linewidth=1
    )

    # Add labels
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(values)*0.02,
            f'{val:.0f}',
            ha='center',
            color='white',
            fontsize=12,
            fontweight='bold'
        )

    # Titles
    ax.set_title(
        title,
        fontsize=15,
        color='white',
        pad=15
    )

    # Styling
    ax.set_facecolor('#111827')

    ax.tick_params(
        colors='white',
        labelsize=11
    )

    # Grid
    ax.grid(
        axis='y',
        linestyle='--',
        alpha=0.2,
        color='white'
    )

    # Border styling
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.spines['left'].set_color('white')
    ax.spines['bottom'].set_color('white')

# Layout
plt.tight_layout()

# Save
plt.savefig(
    'chart6_final_evidence_report.png',
    dpi=240,
    bbox_inches='tight',
    facecolor=fig.get_facecolor()
)

# Show
plt.show()

print("💾 Saved: chart6_final_evidence_report.png")

print()
print("🚨 FINAL AI VERDICT")
print(f"🎯 Primary Suspect: {top_suspect}")
print("📊 The suspect shows behavior far beyond normal employee activity.")

---

## 🎓 WHAT YOU JUST LEARNED

Without realizing it, you used real data science concepts:

| Concept | What we called it |
|---------|------------------|
| `pd.read_csv()` | Loading evidence |
| `.describe()` | Finding what's normal |
| Boolean filtering | Eliminating suspects |
| Bar charts / Heatmaps | Visualizing patterns |
| Weighted scoring | AI decision-making |
| Anomaly detection | Spotting outliers |
| Feature importance | Which clue matters most |

---

> *"Today we identified a criminal using only data.*  
> *Imagine what companies like Google, Netflix, or banks can do with millions of records."*

---